**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 27: DPO 학습 실습 (SFT + DPO)

## 🎯 DPO 학습 실습 개요

이 노트북에서는 **Qwen2.5-1.5B-Instruct** 모델을 사용하여 SFT → DPO 파이프라인을 실습합니다.  
RTX 4060(8GB VRAM) 환경에 최적화된 설정으로 진행합니다.

### 학습 목표
- 🎯 SFT → DPO 2단계 파이프라인 구현
- 1️⃣ FP16 + LoRA로 DPO 학습
- 2️⃣ DPO 핵심 파라미터(beta, loss_type) 이해
- 3️⃣ SFT Only vs SFT+DPO 성능 비교

### GPU 요구사항
- ✅ GPU 필수 (RTX 4060 8GB 이상)
- 모델: Qwen2.5-1.5B-Instruct (FP16)
- 예상 VRAM: ~6-8GB

### 파이프라인 개요
```
Step 1: SFT 학습 → 기본 능력 향상
Step 2: DPO 학습 → 선호도 정렬
```

---


In [1]:
# 필요한 라이브러리 임포트
import torch
import gc
import json
import os
from pathlib import Path
from datetime import datetime

# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print("✅ 기본 라이브러리 임포트 완료")
print(f"📅 실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("초기 상태")
else:
    print("❌ GPU를 사용할 수 없습니다. 이 노트북은 GPU가 필수입니다.")

✅ 기본 라이브러리 임포트 완료
📅 실행 시각: 2026-04-24 11:33:23
🔥 PyTorch: 2.10.0+cu128
🔥 CUDA: 12.8
✅ GPU: NVIDIA GeForce RTX 2070
[초기 상태] GPU: 0.0GB / 7.8GB


In [2]:
# 추가 라이브러리 임포트
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from datasets import Dataset

print("✅ 모든 라이브러리 임포트 완료")
print("   • transformers, peft, trl, datasets")

/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 모든 라이브러리 임포트 완료
   • transformers, peft, trl, datasets


---
## 1️⃣ 환경 설정 및 모델 로드 (Qwen2.5-1.5B-Instruct, FP16)

### 모델 선택 이유
- Qwen2.5-1.5B-Instruct: 한국어 성능이 좋은 소형 모델
- FP16: 양자화 없이 안정적 학습
- LoRA: 학습 파라미터를 최소화하여 메모리 절약


In [3]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("./outputs/dpo_training")
SFT_OUTPUT = OUTPUT_DIR / "sft_checkpoint"
DPO_OUTPUT = OUTPUT_DIR / "dpo_checkpoint"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("📌 모델 설정")
print("=" * 60)
print(f"   모델: {MODEL_NAME}")
print(f"   정밀도: FP16 (양자화 없음)")
print(f"   SFT 출력: {SFT_OUTPUT}")
print(f"   DPO 출력: {DPO_OUTPUT}")

📌 모델 설정
   모델: Qwen/Qwen2.5-1.5B-Instruct
   정밀도: FP16 (양자화 없음)
   SFT 출력: outputs/dpo_training/sft_checkpoint
   DPO 출력: outputs/dpo_training/dpo_checkpoint


In [4]:
# 모델 및 토크나이저 로드
print("=" * 60)
print("📌 모델 로딩")
print("=" * 60)

print("🔄 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # DPO에서는 left padding 권장
print(f"   ✅ 토크나이저 로드 완료 (vocab: {tokenizer.vocab_size})")

print("🔄 모델 로딩... (FP16)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.enable_input_require_grads()
model.config.use_cache = False  # 학습 시 캐시 비활성화

print(f"   ✅ 모델 로드 완료")
print_gpu_memory("모델 로드 후")

📌 모델 로딩
🔄 토크나이저 로딩...
   ✅ 토크나이저 로드 완료 (vocab: 151643)
🔄 모델 로딩... (FP16)


`torch_dtype` is deprecated! Use `dtype` instead!


   ✅ 모델 로드 완료
[모델 로드 후] GPU: 2.9GB / 7.8GB


In [5]:
# LoRA 설정
print("=" * 60)
print("📌 LoRA 설정")
print("=" * 60)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"   LoRA rank (r): {lora_config.r}")
print(f"   LoRA alpha: {lora_config.lora_alpha}")
print(f"   Target modules: {lora_config.target_modules}")
print(f"   Dropout: {lora_config.lora_dropout}")

# 학습 가능 파라미터 확인
peft_model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
print(f"\n📊 파라미터 현황:")
print(f"   학습 가능: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"   전체: {total:,}")
print_gpu_memory("LoRA 적용 후")

# peft_model은 SFT에서 사용할 것이므로 여기서 해제
del peft_model
gc.collect()
torch.cuda.empty_cache()

📌 LoRA 설정
   LoRA rank (r): 16
   LoRA alpha: 32
   Target modules: {'v_proj', 'o_proj', 'k_proj', 'q_proj'}
   Dropout: 0.05

📊 파라미터 현황:
   학습 가능: 4,358,144 (0.28%)
   전체: 1,548,072,448
[LoRA 적용 후] GPU: 2.9GB / 7.8GB


---
## 2️⃣ Step 1: SFT 학습 (기본 능력 향상)

DPO 학습 전에 먼저 SFT로 모델의 기본 능력을 향상시킵니다.

### SFT 데이터
- 한국어 instruction-response 쌍
- Session 26에서 만든 preference 데이터의 chosen 응답을 활용

In [6]:
# SFT 학습 데이터 준비 (26b rejected 데이터 = 저품질 응답)
# → SFT는 "기본적으로 답할 수 있는 수준"만 학습
# → DPO에서 "고품질 응답 스타일"로 정렬
print("=" * 60)
print("📌 SFT 학습 데이터 준비 (저품질 응답으로 기본기 학습)")
print("=" * 60)

rs_data_path = "./output/rejection_sampling_sft/rejection_sampling_data.json"
with open(rs_data_path, "r", encoding="utf-8") as f:
    rs_data = json.load(f)

selected_data = rs_data["selected"]  # 고품질 (DPO chosen용)
rejected_data = rs_data["rejected"]  # 저품질 (SFT 학습용)
print(f"✅ RS 데이터 로드: selected {len(selected_data)}개, rejected {len(rejected_data)}개")
print(f"   SFT 학습: rejected {len(rejected_data)}개 (저품질 → 기본기)")
print(f"   DPO 학습: selected vs rejected (고품질 vs 저품질)")

system_prompt = "당신은 친절하고 도움��� 되는 AI 어시스턴트입니다."

# SFT용: rejected (저품질 응답)으로 학습 → 기본기만 잡기
def format_sft_data(data_list):
    formatted = []
    for item in data_list:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": item["prompt"]},
            {"role": "assistant", "content": item["response"]}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append({"text": text})
    return formatted

sft_formatted = format_sft_data(rejected_data)
sft_dataset = Dataset.from_list(sft_formatted)

print(f"\n✅ SFT 데이터 준비 완료: {len(sft_dataset)}개 샘플")
print(f"\n📋 SFT 학습 프롬프트 (저품질 응답):")
for i, item in enumerate(rejected_data[:5]):
    print(f"   {i+1}. {item['prompt'][:50]} (점수: {item['score']})")

📌 SFT 학습 데이터 준비 (저품질 응답으로 기본기 학습)
✅ RS 데이터 로드: selected 29개, rejected 15개
   SFT 학습: rejected 15개 (저품질 → 기본기)
   DPO 학습: selected vs rejected (고품질 vs 저품질)

✅ SFT 데이터 준비 완료: 15개 샘플

📋 SFT 학습 프롬프트 (저품질 응답):
   1. 머신러닝과 딥러닝의 차이점을 설명하세요. (점수: 3.67)
   2. 좋은 프롬프트를 작성하는 방법을 알려주세요. (점수: 3.67)
   3. Python에서 리스트와 튜플의 차이점은 무엇인가요? (점수: 3.33)
   4. 트랜스포머 모델의 어텐션 메커니즘을 설명하세요. (점수: 3.33)
   5. 오버피팅을 방지하는 방법들을 나열하세요. (점수: 3.67)


In [7]:
# SFT 학습 실행
print("=" * 60)
print("📌 Step 1: SFT 학습 시작")
print("=" * 60)
print_gpu_memory("SFT 학습 전")

# SFT 학습 설정 (RTX 4060 최적화)
sft_config = SFTConfig(
    output_dir=str(SFT_OUTPUT),
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    max_length=1024,
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    report_to="none",
    dataset_text_field="text",
)

# SFT Trainer 생성
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("\n🔥 SFT 학습 시작...")
sft_result = sft_trainer.train()

print(f"\n✅ SFT 학습 완료!")
print(f"   Training Loss: {sft_result.training_loss:.4f}")
print(f"   Steps: {sft_result.global_step}")
print_gpu_memory("SFT 학습 후")


📌 Step 1: SFT 학습 시작
[SFT 학습 전] GPU: 2.9GB / 7.8GB


/usr/bin/ld: cannot find -lcufile: 그런 파일이나 디렉터리가 없습니다
collect2: error: ld returned 1 exit status
/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating train dataset: 100%|██████████| 15/15 [00:00<00:00, 5492.32 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the t


🔥 SFT 학습 시작...


Step,Training Loss
1,1.577800
2,1.514100
3,1.529900
4,1.445600
5,1.478600
6,1.251100
7,1.288200
8,1.282000
9,1.215600
10,1.196000



✅ SFT 학습 완료!
   Training Loss: 1.2193
   Steps: 20
[SFT 학습 후] GPU: 2.9GB / 7.8GB


In [8]:
# SFT 모델 저장 + 비교용 응답 미리 생성
print("🔄 SFT 모델 저장 중...")
sft_trainer.save_model(str(SFT_OUTPUT))
tokenizer.save_pretrained(str(SFT_OUTPUT))
print(f"✅ SFT 모델 저장 완료: {SFT_OUTPUT}")

print("=" * 60)
print("📌 SFT 모델 응답 저장 (DPO 비교용)")
print("=" * 60)

sft_trainer.model.config.use_cache = True
sft_trainer.model.eval()
for name, param in sft_trainer.model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.float16)

# 비교 프롬프트 (학습 데이터에서 선택)
comparison_prompts = [
    "머신러닝과 딥러닝의 차이점을 설명하세요.",
    "트랜스포머 모델의 어텐션 메커니즘을 설명하세요.",
    "LoRA 파인튜닝의 장점을 설명하세요.",
    "오버피팅을 방지하는 방법들을 나열하세요.",
]

def generate_response(model, prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

sft_responses = []
for prompt in comparison_prompts:
    resp = generate_response(sft_trainer.model, prompt)
    sft_responses.append(resp)
    print(f"\n❓ {prompt}")
    print(f"🤖 SFT: {resp[:300]}")
    print("-" * 50)

print(f"\n✅ SFT 응답 {len(sft_responses)}개 저장 완료")

🔄 SFT 모델 저장 중...
✅ SFT 모델 저장 완료: outputs/dpo_training/sft_checkpoint
📌 SFT 모델 응답 저장 (DPO 비교용)

❓ 머신러닝과 딥러닝의 차이점을 설명하세요.
🤖 SFT: 머신러닝(Machine Learning)과 딥러닝(Deep Learning)은 모두 컴퓨터 시스템에서 데이터를 학습하고 예측하는 기술로, 서로 관련성이 있지만 구체적인 차이점이 있습니다.

1. **학습 방법**:
   - **머신러닝**: 이는 단계적으로 데이터를 분석하여 모델을 훈련시키는 방식으로, 일반적으로 규칙적이고 선형적인 관계를 가진 문제에 더 적합합니다.
   - **딥러닝**: 이는 복잡한 신경망을 사용하여 데이터로부터 깊게 추론하는 방식으로, 특히 비선형 및 복잡한 패턴을 파악하는데 효과적입니다.

2. **구조**
--------------------------------------------------

❓ 트랜스포머 모델의 어텐션 메커니즘을 설명하세요.
🤖 SFT: 트랜스포머 모델은 특히 인공지능 언어 처리에 중요한 역할을 하는 모델로, 주요한 특징 중 하나가 어텐션 메커니즘이라는 것입니다.

1. **어텐션 메커니즘 (Attention Mechanism)**:
   - 어텐션 메커니즘은 각 토큰(단어) 또는 임베딩 벡터 간의 상호작용을 측정하는 방법입니다.
   - 이 메커니즘이란, 특정 토큰이 다른 토큰에 대해 얼마나 중요하게 생각하는지를 나타내는 기법입니다.

2. **원리**:
   - 모든 토큰들이 모두 같은 가중치를 가지도록 하여 모든 정보를 동등히 고려하도록 합니다.
   - 하지
--------------------------------------------------

❓ LoRA 파인튜닝의 장점을 설명하세요.
🤖 SFT: LoRA (Low-Rank Adaptation) 파인튜닝은 대규모 데이터셋을 사용하여 모델을 훈련하는 데 필요한 시간과 computational resources를 줄이는 방법 중

In [9]:
# GPU 메모리 정리 (DPO 학습 준비)
print("🧹 SFT 학습 메모리 정리...")
print_gpu_memory("정리 전")

del sft_trainer
del model
gc.collect()
torch.cuda.empty_cache()

print_gpu_memory("정리 후")
print("✅ 메모리 정리 완료 → DPO 학습 준비")

🧹 SFT 학습 메모리 정리...
[정리 전] GPU: 2.9GB / 7.8GB
[정리 후] GPU: 0.0GB / 7.8GB
✅ 메모리 정리 완료 → DPO 학습 준비


---
## 3️⃣ Preference 데이터 로드 및 포맷팅

Session 26에서 만든 선호도 데이터를 DPO 학습 형식으로 로드합니다.

In [10]:
# DPO Preference 데이터 구성 (26b RS 데이터 활용)
print("=" * 60)
print("📌 DPO Preference 데이터 구성")
print("=" * 60)

# 프롬프트별 best(selected) vs worst(rejected) 매칭
# selected에서 프롬프트별 최고 점수, rejected에서 최저 점수
best_by_prompt = {}
for item in selected_data:
    p = item["prompt"]
    if p not in best_by_prompt or item["score"] > best_by_prompt[p]["score"]:
        best_by_prompt[p] = item

worst_by_prompt = {}
for item in rejected_data:
    worst_by_prompt[item["prompt"]] = item

# chosen/rejected 쌍 구성
raw_preference_data = []
for prompt in best_by_prompt:
    if prompt in worst_by_prompt:
        raw_preference_data.append({
            "prompt": prompt,
            "chosen": best_by_prompt[prompt]["response"],
            "rejected": worst_by_prompt[prompt]["response"],
        })

print(f"✅ DPO Preference 쌍: {len(raw_preference_data)}개")
for i, item in enumerate(raw_preference_data[:3]):
    print(f"\n  {i+1}. {item['prompt'][:50]}")
    print(f"     chosen:   {item['chosen'][:60]}...")
    print(f"     rejected: {item['rejected'][:60]}...")

📌 DPO Preference 데이터 구성
✅ DPO Preference 쌍: 15개

  1. 머신러닝과 딥러닝의 차이점을 설명하세요.
     chosen:   머신러닝과 딥러닝은 모두 컴퓨터에서 데이터를 분석하고, 모델을 학습하는 기술로 서로 관련성이 큽니다. 그러나...
     rejected: 머신러닝(Machine Learning)과 딥러닝(Depth Learning)은 컴퓨터과학에서 매우 중요한 ...

  2. 좋은 프롬프트를 작성하는 방법을 알려주세요.
     chosen:   작성하는 프롬프트의 좋은 기법과 팁 몇 가지:

1) 명확성을 고려하세요: 사용자가 원하시는 정보를 이해할 ...
     rejected: 작성할 프롬프트를 위한 몇 가지 좋은 아이디어는 다음과 같습니다:

1. 특정 주제를 대상으로 한 프롬프트:...

  3. Python에서 리스트와 튜플의 차이점은 무엇인가요?
     chosen:   Python에서는 리스트(list)와 튜플(tuple) 두 가지 자료형이 있습니다.

1. **리스트(Lis...
     rejected: Python에서 리스트(list)와 튜플(tuple) 모두 데이터를 관리하는데 사용되지만, 두 가지 간에 중...


In [11]:
# DPO 형식으로 변환
print("=" * 60)
print("📌 DPO 학습 형식으로 데이터 변환")
print("=" * 60)

system_prompt = "당신은 친절하고 도움이 되는 AI 어시스턴트입니다."

def format_for_dpo(data_list):
    """Preference 데이터를 DPO 형식으로 변환"""
    formatted = []
    for item in data_list:
        formatted.append({
            "chosen": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["chosen"]}
            ],
            "rejected": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["rejected"]}
            ]
        })
    return formatted

dpo_data = format_for_dpo(raw_preference_data)
dpo_dataset = Dataset.from_list(dpo_data)

print(f"✅ DPO 데이터셋 생성 완료: {len(dpo_dataset)}개 샘플")
print(f"📋 컬럼: {dpo_dataset.column_names}")
print(f"\n📋 첫 번째 샘플:")
print(f"   Chosen: {dpo_dataset[0]['chosen'][-1]['content'][:100]}...")
print(f"   Rejected: {dpo_dataset[0]['rejected'][-1]['content'][:100]}...")

📌 DPO 학습 형식으로 데이터 변환
✅ DPO 데이터셋 생성 완료: 15개 샘플
📋 컬럼: ['chosen', 'rejected']

📋 첫 번째 샘플:
   Chosen: 머신러닝과 딥러닝은 모두 컴퓨터에서 데이터를 분석하고, 모델을 학습하는 기술로 서로 관련성이 큽니다. 그러나 주요 차이는 다음과 같습니다:

1. **학습 과정**: 
   - *...
   Rejected: 머신러닝(Machine Learning)과 딥러닝(Depth Learning)은 컴퓨터과학에서 매우 중요한 개념으로, 둘 다 인공지능(AI)에 있어 중요한 역할을 합니다. 그러나 ...


---
## 4️⃣ Step 2: DPO 학습 (DPOTrainer from trl)

SFT 체크포인트 위에 DPO 학습을 진행합니다.

### DPO 학습 흐름
```
1. SFT 체크포인트 로드 (reference model + policy model)
2. Preference 데이터로 DPO 손실 계산
3. Policy model만 업데이트 (reference는 고정)
4. chosen 확률 ↑, rejected 확률 ↓
```

In [12]:
# DPO용 모델 다시 로드
print("=" * 60)
print("📌 DPO 학습을 위한 모델 재로드")
print("=" * 60)

# 기본 모델 로드 (FP16)
print("🔄 Base 모델 로딩...")
dpo_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)
dpo_model.enable_input_require_grads()
dpo_model.config.use_cache = False

# SFT LoRA 가중치 로드
if SFT_OUTPUT.exists():
    print("🔄 SFT LoRA 가중치 로드...")
    dpo_model = PeftModel.from_pretrained(
        dpo_model,
        str(SFT_OUTPUT),
        is_trainable=True,
    )
    print("   ✅ SFT 가중치 로드 완료")
else:
    print("⚠️ SFT 체크포인트가 없습니다. Base 모델로 진행합니다.")
    dpo_model = get_peft_model(dpo_model, lora_config)

print_gpu_memory("DPO 모델 로드 후")

📌 DPO 학습을 위한 모델 재로드
🔄 Base 모델 로딩...
🔄 SFT LoRA 가중치 로드...
   ✅ SFT 가중치 로드 완료
[DPO 모델 로드 후] GPU: 2.9GB / 7.8GB


In [13]:
# DPO 학습 설정 및 실행
print("=" * 60)
print("📌 DPO 학습 시작")
print("=" * 60)

# DPO 설정 (RTX 4060 최적화)
dpo_config = DPOConfig(
    output_dir=str(DPO_OUTPUT),
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    fp16=True,
    max_length=1024,
    max_prompt_length=512,
    beta=0.3,                  # DPO 핵심 파라미터
    loss_type="sigmoid",       # DPO 손실 타입
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    report_to="none",
    remove_unused_columns=False,
)

print("📋 DPO 학습 설정:")
print(f"   beta: {dpo_config.beta}")
print(f"   loss_type: {dpo_config.loss_type}")
print(f"   batch_size: {dpo_config.per_device_train_batch_size}")
print(f"   gradient_accumulation: {dpo_config.gradient_accumulation_steps}")
print(f"   learning_rate: {dpo_config.learning_rate}")
print(f"   max_length: {dpo_config.max_length}")
print(f"   epochs: {dpo_config.num_train_epochs}")

# DPO Trainer 생성
dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("\n🔥 DPO 학습 시작...")
print_gpu_memory("DPO 학습 시작")

dpo_result = dpo_trainer.train()

print(f"\n✅ DPO 학습 완료!")
print(f"   Training Loss: {dpo_result.training_loss:.4f}")
print(f"   Steps: {dpo_result.global_step}")
print_gpu_memory("DPO 학습 후")


📌 DPO 학습 시작
📋 DPO 학습 설정:
   beta: 0.3
   loss_type: sigmoid
   batch_size: 1
   gradient_accumulation: 8
   learning_rate: 0.0001
   max_length: 1024
   epochs: 10


Tokenizing train dataset: 100%|██████████| 15/15 [00:00<00:00, 529.50 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



🔥 DPO 학습 시작...
[DPO 학습 시작] GPU: 2.9GB / 7.8GB


/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
1,11.335900
2,10.375800
3,11.343200
4,7.415200
5,9.398800
6,5.673800
7,6.150300
8,2.983100
9,3.712200
10,1.155800


/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_valid


✅ DPO 학습 완료!
   Training Loss: 3.6965
   Steps: 20
[DPO 학습 후] GPU: 2.9GB / 7.8GB


In [14]:
# DPO 모델 저장
print("🔄 DPO 모델 저장 중...")
dpo_trainer.save_model(str(DPO_OUTPUT))
tokenizer.save_pretrained(str(DPO_OUTPUT))
print(f"✅ DPO 모델 저장 완료: {DPO_OUTPUT}")

🔄 DPO 모델 저장 중...
✅ DPO 모델 저장 완료: outputs/dpo_training/dpo_checkpoint


---
## 5️⃣ DPO 학습 파라미터 설명 (beta, loss_type)

### 핵심 파라미터

| 파라미터 | 값 | 설명 |
|---------|-----|------|
| `beta` | 0.1 | KL 페널티 강도. 높을수록 보수적 학습 |
| `loss_type` | sigmoid | DPO 손실 함수 종류 |
| `max_length` | 1024 | chosen/rejected 최대 토큰 수 |
| `max_prompt_length` | 512 | 프롬프트 최대 토큰 수 |

### beta 값에 따른 효과

```
beta = 0.01 → 매우 공격적 학습 (큰 변화, 불안정할 수 있음)
beta = 0.1  → 일반적 설정 (균형잡힌 학습)
beta = 0.5  → 매우 보수적 학습 (작은 변화, 안정적)
```

### loss_type 종류

| 타입 | 설명 |
|------|------|
| `sigmoid` | 기본 DPO 손실 (가장 일반적) |
| `hinge` | SVM 스타일 힌지 손실 |
| `ipo` | Identity Preference Optimization |
| `kto_pair` | Kahneman-Tversky Optimization |

In [15]:
# DPO 파라미터 영향 분석
print("=" * 60)
print("📌 DPO 핵심 파라미터 분석")
print("=" * 60)

import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# beta 값에 따른 그래디언트 크기 비교
log_ratio_diff = np.linspace(-3, 3, 100)

print("\n📊 beta 값에 따른 학습 강도:")
print("-" * 50)

for beta in [0.01, 0.1, 0.5]:
    # DPO 그래디언트 크기 (대략적)
    grad_magnitude = np.mean(np.abs(beta * (1 - sigmoid(beta * log_ratio_diff))))
    
    label = ""
    if beta == 0.01:
        label = "(공격적 - 큰 변화)"
    elif beta == 0.1:
        label = "(균형 - 권장값)"
    else:
        label = "(보수적 - 작은 변화)"
    
    bar = "█" * int(grad_magnitude * 100)
    print(f"   beta={beta:>4}: grad~{grad_magnitude:.4f} {bar} {label}")

print("\n💡 RTX 4060 환경에서는 beta=0.1이 안정적인 기본값입니다.")
print("   데이터가 적을 때는 beta를 높이고, 많을 때는 낮출 수 있습니다.")

📌 DPO 핵심 파라미터 분석

📊 beta 값에 따른 학습 강도:
--------------------------------------------------
   beta=0.01: grad~0.0050  (공격적 - 큰 변화)
   beta= 0.1: grad~0.0500 █████ (균형 - 권장값)
   beta= 0.5: grad~0.2500 █████████████████████████ (보수적 - 작은 변화)

💡 RTX 4060 환경에서는 beta=0.1이 안정적인 기본값입니다.
   데이터가 적을 때는 beta를 높이고, 많을 때는 낮출 수 있습니다.


---
## 6️⃣ 학습 전후 비교 (SFT Only vs SFT+DPO)

SFT만 적용한 모델과 SFT+DPO를 적용한 모델의 응답을 비교합니다.

In [16]:
# SFT vs DPO 모델 비교
print("=" * 60)
print("📊 SFT Only vs SFT+DPO 비교")
print("=" * 60)

dpo_trainer.model.config.use_cache = True
dpo_trainer.model.eval()
for name, param in dpo_trainer.model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.float16)

dpo_responses = []
for i, prompt in enumerate(comparison_prompts):
    dpo_resp = generate_response(dpo_trainer.model, prompt)
    dpo_responses.append(dpo_resp)
    
    print(f"\n{'='*60}")
    print(f"❓ 질문 {i+1}: {prompt}")
    print(f"\n🔵 SFT Only:")
    print(f"   {sft_responses[i][:350]}")
    print(f"\n🟢 SFT+DPO:")
    print(f"   {dpo_resp[:350]}")

print(f"\n{'='*60}")
print("✅ 비교 완료")

📊 SFT Only vs SFT+DPO 비교

❓ 질문 1: 머신러닝과 딥러닝의 차이점을 설명하세요.

🔵 SFT Only:
   머신러닝(Machine Learning)과 딥러닝(Deep Learning)은 모두 컴퓨터 시스템에서 데이터를 학습하고 예측하는 기술로, 서로 관련성이 있지만 구체적인 차이점이 있습니다.

1. **학습 방법**:
   - **머신러닝**: 이는 단계적으로 데이터를 분석하여 모델을 훈련시키는 방식으로, 일반적으로 규칙적이고 선형적인 관계를 가진 문제에 더 적합합니다.
   - **딥러닝**: 이는 복잡한 신경망을 사용하여 데이터로부터 깊게 추론하는 방식으로, 특히 비선형 및 복잡한 패턴을 파악하는데 효과적입니다.

2. **구조**:
   - **머신러닝**: 일반적으로 회귀나 분류 문제에 적용되며, 이러한 문제들은 기

🟢 SFT+DPO:
   머신러닝(Machine Learning)과 딥러닝(Deep Learning)은 모두 컴퓨터가 데이터를 학습하고 이를 통해 새로운 정보를 얻을 수 있도록 하는 기술로, 서로 관련성이 있지만 구체적인 차이점이 있습니다.

1. **학습 방식**:
   - **머신러닝**: 이는 단순히 특정 알고리즘에 대한 입력 데이터를 사용하여 그 결과를 예측하거나 분류하는 것을 목표로 합니다.
   - **딥러닝**: 더 복잡한 모델을 사용하여 데이터로부터 깊은 의미를 추출하고 이를 이용해 새로운 정보를 생성하는 방식입니다.

2. **특성 개수와 깊이**:
   - **머신러닝**: 일반적으로 한 번에 하나 이상의 특성을 사용합니다.


❓ 질문 2: 트랜스포머 모델의 어텐션 메커니즘을 설명하세요.

🔵 SFT Only:
   트랜스포머 모델은 특히 인공지능 언어 처리에 중요한 역할을 하는 모델로, 주요한 특징 중 하나가 어텐션 메커니즘이라는 것입니다.

1. **어텐션 메커니즘 (Attention Mechanism)**:
   - 어텐션 메커니즘은 각 토큰(단어) 또는 임베딩 벡터 간의 상호작용을 측정하

---
## 7️⃣ 선호도 정렬 평가

DPO 학습이 실제로 선호도 정렬에 효과가 있는지 정량적으로 평가합니다.

In [17]:
# LLM Judge로 SFT vs DPO 정량 비교
print("=" * 60)
print("📊 LLM Judge 정량 비교 (SFT vs DPO)")
print("=" * 60)

from openai import OpenAI
client = OpenAI()

def judge_response(prompt, response):
    """GPT-4o-mini로 응답 품질 평가 (1-5점)"""
    judge_prompt = f"""AI 응답을 평가하세요.

질문: {prompt}
응답: {response}

평가 기준 (각 1-5점):
1. 정확성: 내용이 올바른가?
2. 완성도: 충분히 상세한가?
3. 명확성: 구조적이고 이해하기 쉬운가?

JSON으로만 답변: {{"accuracy": 점수, "completeness": 점수, "clarity": 점수}}"""
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
        result = json.loads(resp.choices[0].message.content)
        avg = (result["accuracy"] + result["completeness"] + result["clarity"]) / 3
        return round(avg, 2)
    except Exception as e:
        print(f"  ⚠️ 평가 실패: {e}")
        return 3.0

sft_scores = []
dpo_scores = []

for i, prompt in enumerate(comparison_prompts):
    s_score = judge_response(prompt, sft_responses[i])
    d_score = judge_response(prompt, dpo_responses[i])
    sft_scores.append(s_score)
    dpo_scores.append(d_score)
    
    diff = d_score - s_score
    emoji = "📈" if diff > 0 else ("📉" if diff < 0 else "➡️")
    print(f"\n{emoji} 질문 {i+1}: {prompt[:40]}...")
    print(f"   SFT: {s_score:.1f}점 / DPO: {d_score:.1f}점 ({diff:+.1f})")

print(f"\n{'='*60}")
print(f"📊 평균 점수:")
print(f"   SFT Only: {sum(sft_scores)/len(sft_scores):.2f}")
print(f"   SFT+DPO:  {sum(dpo_scores)/len(dpo_scores):.2f}")
print(f"   변화:     {sum(dpo_scores)/len(dpo_scores) - sum(sft_scores)/len(sft_scores):+.2f}")

📊 LLM Judge 정량 비교 (SFT vs DPO)

➡️ 질문 1: 머신러닝과 딥러닝의 차이점을 설명하세요....
   SFT: 3.7점 / DPO: 3.7점 (+0.0)

📉 질문 2: 트랜스포머 모델의 어텐션 메커니즘을 설명하세요....
   SFT: 3.7점 / DPO: 2.7점 (-1.0)

➡️ 질문 3: LoRA 파인튜닝의 장점을 설명하세요....
   SFT: 3.7점 / DPO: 3.7점 (+0.0)

📉 질문 4: 오버피팅을 방지하는 방법들을 나열하세요....
   SFT: 4.3점 / DPO: 4.0점 (-0.3)

📊 평균 점수:
   SFT Only: 3.83
   SFT+DPO:  3.50
   변화:     -0.33


In [18]:
# DPO 학습 로그 분석
print("=" * 60)
print("📌 DPO 학습 로그 분석")
print("=" * 60)

# 학습 이력에서 주요 메트릭 추출
if hasattr(dpo_trainer, 'state') and dpo_trainer.state.log_history:
    log_history = dpo_trainer.state.log_history
    
    print("\n📊 학습 이력:")
    for entry in log_history:
        if 'loss' in entry:
            step = entry.get('step', '?')
            loss = entry.get('loss', '?')
            lr = entry.get('learning_rate', '?')
            print(f"   Step {step}: loss={loss:.4f}" if isinstance(loss, float) else f"   Step {step}: loss={loss}")
    
    # DPO 특유의 메트릭
    print("\n📊 DPO 메트릭 설명:")
    print("   • loss: DPO 손실 (낮을수록 좋음)")
    print("   • rewards/chosen: chosen 응답에 대한 암시적 보상")
    print("   • rewards/rejected: rejected 응답에 대한 암시적 보상")
    print("   • rewards/margins: chosen - rejected (양수가 좋음)")
else:
    print("\n⚠️ 학습 로그를 찾을 수 없습니다.")

📌 DPO 학습 로그 분석

📊 학습 이력:
   Step 1: loss=11.3359
   Step 2: loss=10.3758
   Step 3: loss=11.3432
   Step 4: loss=7.4152
   Step 5: loss=9.3988
   Step 6: loss=5.6738
   Step 7: loss=6.1503
   Step 8: loss=2.9831
   Step 9: loss=3.7122
   Step 10: loss=1.1558
   Step 11: loss=1.7968
   Step 12: loss=0.5518
   Step 13: loss=0.7297
   Step 14: loss=0.3160
   Step 15: loss=0.3319
   Step 16: loss=0.1574
   Step 17: loss=0.0480
   Step 18: loss=0.2280
   Step 19: loss=0.0854
   Step 20: loss=0.1414

📊 DPO 메트릭 설명:
   • loss: DPO 손실 (낮을수록 좋음)
   • rewards/chosen: chosen 응답에 대한 암시적 보상
   • rewards/rejected: rejected 응답에 대한 암시적 보상
   • rewards/margins: chosen - rejected (양수가 좋음)


In [19]:
# GPU 메모리 정리
print("🧹 GPU 메모리 최종 정리...")
print_gpu_memory("정리 전")

del dpo_trainer
del dpo_model
gc.collect()
torch.cuda.empty_cache()

print_gpu_memory("정리 후")
print("✅ 메모리 정리 완료")

🧹 GPU 메모리 최종 정리...
[정리 전] GPU: 2.9GB / 7.8GB
[정리 후] GPU: 0.0GB / 7.8GB
✅ 메모리 정리 완료


---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **SFT → DPO 파이프라인**: 2단계로 모델을 점진적으로 개선

2️⃣ **4bit + LoRA**: RTX 4060(8GB)에서 DPO 학습 가능

3️⃣ **DPO 핵심 파라미터**:
   - `beta=0.1`: KL 페널티 강도 (선호도 학습 강도 조절)
   - `loss_type='sigmoid'`: 기본 DPO 손실 함수

4️⃣ **DPO의 효과**:
   - Chosen 스타일(구조적, 상세, 친절)로 답변 유도
   - Rejected 스타일(짧고 불친절) 답변 확률 감소

5️⃣ **RTX 4060 설정**:
   - `per_device_train_batch_size=1`
   - `gradient_accumulation_steps=8`
   - `fp16=True`
   - `max_seq_length=1024`

### 생성된 파일

| 파일 | 설명 |
|------|------|
| `outputs/dpo_training/sft_checkpoint/` | SFT 모델 체크포인트 |
| `outputs/dpo_training/dpo_checkpoint/` | DPO 모델 체크포인트 |

### 다음 세션 예고

- 📘 **Session 28**: DeepSeek R1 Case Study with GRPO
  - GRPO 알고리즘으로 수학 추론 능력 향상

In [20]:
# 최종 확인
print("=" * 60)
print("📌 최종 확인")
print("=" * 60)

# 저장된 체크포인트 확인
for checkpoint_dir in [SFT_OUTPUT, DPO_OUTPUT]:
    if checkpoint_dir.exists():
        files = list(checkpoint_dir.glob("*"))
        total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
        print(f"  ✅ {checkpoint_dir.name}: {len(files)} 파일, {total_size:.1f}MB")
    else:
        print(f"  ⚠️ {checkpoint_dir.name}: 디렉토리 없음")

print("\n" + "=" * 60)
print("✅ Session 27 완료!")
print("   SFT → DPO 파이프라인을 성공적으로 실습했습니다.")
print("   다음 세션에서 GRPO를 활용한 추론 모델 학습을 진행합니다.")

📌 최종 확인
  ✅ sft_checkpoint: 21 파일, 31.8MB
  ✅ dpo_checkpoint: 21 파일, 31.8MB

✅ Session 27 완료!
   SFT → DPO 파이프라인을 성공적으로 실습했습니다.
   다음 세션에서 GRPO를 활용한 추론 모델 학습을 진행합니다.
